# 크레온 플러스 1분봉 수집 (전 종목)

KOSPI·KOSDAQ **보통주 전 종목**의 KRX 정규장 1분봉과 NXT 1분봉을 크레온 플러스에서 받을 수 있는 만큼 과거까지 받는 노트북입니다. **윈도우 전용**이고, 위에서부터 순서대로 실행하면 됩니다.

**단계**
1. 실행 환경 점검 (32비트 Python, 관리자 권한, 크레온 연결, 디스크 여유 공간)
2. 종목 목록: 보통주만 고르고 `universe.csv`로 고정
3. 공통 요청 함수
4. 전 종목 수집: KRX·NXT × 수정주가/무수정주가 (**30시간 안팎**)
5. 수집 결과 점검
6. 시작일 맞추기 (단독 실행)

**결과**: `creon_1min/snapshots/all_stocks/` 아래에 저장됩니다. 파일별 형식은 바로 아래 **생성 파일 정리**를 보세요.

**기간 고정**: 모든 종목·데이터를 `START_DATE`(2024-10-14)부터 `END_DATE`(2026-11-02)까지만 저장합니다. 범위 밖의 봉은 받더라도 버립니다.

**여러 번에 나눠 실행**: 날짜가 바뀌어도 같은 폴더에서 이어 받고, 이미 받은 파일은 건너뜁니다. 크레온 연결이 끊기거나 실패가 연달아 나면 수집이 멈추니, 다시 로그인하고 처음부터 다시 실행하면 이어서 받습니다.

**실행 전 준비**
- 크레온 플러스와 VS Code를 모두 **관리자 권한**으로 실행. SSH 연결이 아닌 **윈도우 화면에서** 실행
- 오래 걸리므로 윈도우 **절전 모드를 끄기** (설정 → 시스템 → 전원)
- **32비트 Python 3.11** 환경에 아래 버전으로 설치. 32비트용 빌드가 있는 조합으로 고정한 것입니다.

```
pip install pywin32 "pandas==2.0.3" "numpy<2" "psutil<7.1.2" ipykernel
```

## 생성 파일 정리

이 노트북을 실행하면 아래 파일이 생깁니다. 기준: 2026-11-03 수집 완료, 시작일 맞추기 실행 후.

```
creon_1min/
├── .gitignore                    # 폴더 전체를 git에서 제외 ("*")
└── snapshots/all_stocks/
    ├── adjusted/<종목코드>.csv.gz      # KRX 정규장, 수정주가     2,539개
    ├── raw/<종목코드>.csv.gz           # KRX 정규장, 무수정주가   2,539개
    ├── nxt_adjusted/<종목코드>.csv.gz  # NXT, 수정주가            603개
    ├── nxt_raw/<종목코드>.csv.gz       # NXT, 무수정주가          603개
    ├── universe.csv              # 종목 목록과 보통주 판정
    ├── manifest.jsonl            # 파일별 수집 기록
    ├── collection_report.txt     # 실행 로그
    ├── check_KRX.csv             # KRX 종목별 점검표
    └── check_NXT.csv             # NXT 종목별 점검표
```

### 1분봉 데이터 (`<종류>/<종목코드>.csv.gz`)

| 폴더 | 거래소 | 가격 | 종목 수 | 기간 | 시각 |
|---|---|---|---|---|---|
| `adjusted/` | KRX 정규장 | 수정주가 | 2,539 (보통주 전체) | 2024-10-14 ~ 2026-11-02 | 09:01 ~ 15:30 (지연 개장일은 최대 16:30) |
| `raw/` | KRX 정규장 | 무수정주가 | 2,539 | 같음 | 같음 |
| `nxt_adjusted/` | NXT 프리·메인·애프터 | 수정주가 | 603 (NXT 거래 가능 종목) | 종목마다 다름 (가장 이른 2025-03-04) ~ 2026-11-02 | 08:01 ~ 20:00 |
| `nxt_raw/` | NXT 프리·메인·애프터 | 무수정주가 | 603 | 같음 | 같음 |

- 열: `date,time,open,high,low,close,volume,value`. 전부 정수입니다.
  - `date`: `YYYYMMDD` (예: `20241014`)
  - `time`: `HHMM`, 봉이 끝나는 시각 (`901` = 09:00~09:01 봉). 앞자리 0이 없으므로 `date * 10000 + time`으로 정렬 키를 만듭니다.
  - `volume`: 거래량(주), `value`: 거래대금(원, 대략 `close × volume`)
- 날짜·시각 오름차순으로 저장되어 있습니다.
- 수정주가는 가격·거래량이 반올림되어 있고 이후 권리 변동이 생기면 값이 바뀝니다. 원래 값이 필요하면 무수정(`raw`)을 쓰세요. 같은 종목의 수정/무수정은 `date, time`으로 1:1 대응합니다.
- 봉이 하나도 없는 KRX 종목 15개는 헤더만 있는 빈 파일입니다 (거래정지 등).
- 같은 시각 NXT와 KRX 종가 차이는 중앙값 6.4bp, 최대 약 21bp입니다.
- 읽을 때: `pd.read_csv(path)` (gzip은 확장자로 자동 인식).

### 데이터를 쓸 때 알아 둘 점

- **봉 시각은 끝 시각 기준입니다.** `901`은 09:00:00~09:00:59 봉입니다. 수업 DB의 키움 분봉처럼 시작 시각 기준으로 쓰려면 1분을 빼세요. 이렇게 맞추면 키움 분봉과 시가·고가·저가·종가가 모두 일치합니다 (13종목, 2025-10~2026-10에서 확인).
  ```python
  bar_start = pd.to_datetime(df["date"] * 10000 + df["time"], format="%Y%m%d%H%M") - pd.Timedelta(minutes=1)
  ```
- **KRX 하루 구성**: 09:01~15:20 봉과 종가 단일가 봉 `1530` 하나(15:21~15:29 봉 없음)로 하루 최대 381개입니다. 수능일은 10:01~16:30, 1월 2일은 10:01부터이고, 서킷브레이커 날은 약 30분 동안 봉이 비어 있습니다. 시각을 `<= 1530`으로 자르면 수능일 봉이 잘립니다.
- **체결이 없는 분은 봉이 없습니다.** 빈 분을 채우지 않았으므로, 분 단위로 맞춰 쓰려면 직접 채워야 합니다. 거래가 적은 종목일수록 많이 비어 있습니다.
- **수정/무수정**: 같은 종목의 두 파일은 `date, time`이 행 단위로 1:1 일치합니다 (전 종목 확인). `value`는 두 파일에서 전 종목 같고, `volume`은 권리 변동이 있는 종목(KRX 534개, NXT 52개)에서만 다릅니다.
- **`value` 단위**: 원이며 1만 원 단위로 반올림되어 있습니다. 거래량 1주처럼 작은 봉은 0일 수 있습니다.
- **15:30 봉 거래량**: 애프터마켓 포함 옵션으로 받은 값보다 약간 큽니다 (리노공업 기준 전체 거래량의 0.06%, 가격은 같음). 원인은 확인하지 못했습니다.
- **거래가 끊긴 종목**: 봉이 있는 KRX 2,524종목 중 93종목은 마지막 날짜가 2026-11-02보다 이릅니다 (거래정지 등). 백테스트에서 거래 가능 여부를 따로 처리해야 합니다.
- **생존 편향**: 수집 시점에 상장된 종목만 대상이라, 기간 중 상장폐지된 종목은 들어 있지 않습니다.
- **검증**: 6,284개 파일을 모두 읽어 결측, 중복, 정렬, 가격 오류가 없음을 확인했습니다. 다른 날 받은 15종목 스냅샷(`snapshots/20261102/`)과 겹치는 봉은 값이 모두 같습니다.

### `universe.csv` — 종목 목록

처음 실행할 때 크레온에서 받아 고정한 KOSPI·KOSDAQ 전체 4,309개 종목입니다. 수집 대상은 `common == True`인 2,539개입니다.

| 열 | 내용 |
|---|---|
| `code` | 종목코드 6자리. **`dtype={"code": str}`로 읽어야** 앞자리 0이 유지됩니다 |
| `name`, `market` | 종목명, `KOSPI`/`KOSDAQ` |
| `section` | 크레온 부 구분 (1 주권, 10 ETF, 17 ETN, 3 리츠 등) |
| `listed_date` | 상장일 `YYYYMMDD` |
| `status` | 0 정상, 1 거래정지, 2 거래중단 (목록을 받은 시점 기준) |
| `nxt` | NXT 거래 가능 여부 (1/0) |
| `common` | 보통주 판정 결과 (주권 + 코드 끝자리 0 + 이름에 '스팩' 없음) |

### `manifest.jsonl` — 파일별 수집 기록

데이터 파일 하나를 저장할 때마다 한 줄(JSON)씩 추가됩니다. 같은 작업을 다시 받으면 줄이 더 붙으므로 `(code, kind)`별 마지막 줄을 쓰세요.

| 키 | 내용 |
|---|---|
| `code`, `kind` | 종목코드, 폴더명 (`adjusted`/`raw`/`nxt_adjusted`/`nxt_raw`) |
| `rows`, `first_date`, `last_date` | 저장한 봉 수와 기간 |
| `requests`, `received`, `max_bars_per_request` | 요청 횟수, 받은 봉 수 (기간 자르기 전), 요청당 최대 봉 수 |
| `stop_reason` | 수집 종료 사유. `continue_0` 서버가 끝을 알림(정상), `empty_page` 빈 응답, `no_older_data` 더 과거 없음 |
| `seconds`, `saved_at` | 걸린 시간(초), 저장 시각 |

⚠️ "6. 시작일 맞추기"는 manifest를 고치지 않습니다. 2024-10-11 봉을 지운 KRX 파일 667개(adjusted 334, raw 333)는 manifest의 `rows`, `first_date`가 지우기 전 값입니다. 정확한 값은 파일을 직접 세세요.

### `collection_report.txt` — 실행 로그

셀 출력이 그대로 이어서 기록됩니다 (`===== 실행 YYYY-MM-DD HH:MM =====`으로 실행 구분). 환경 점검 결과, 종목별 완료·실패, 진행률, 점검 요약, 시작일 맞추기 결과가 들어 있습니다. 실패 원인을 찾을 때 `실패 `로 시작하는 줄을 보면 됩니다.

### `check_KRX.csv`, `check_NXT.csv` — 종목별 점검표

"5. 수집 결과 점검" 셀이 만듭니다. 무수정(`raw`) 파일 기준 종목당 한 줄이고, `code`는 `universe.csv`처럼 문자열로 읽으세요.

| 열 | 내용 |
|---|---|
| `stop`, `requests`, `rows` | 무수정 파일의 종료 사유, 요청 횟수, 봉 수 |
| `first`, `last` | 첫·마지막 봉 `YYYYMMDDHHMM` |
| `days`, `median_bars` | 거래일 수, 하루 봉 수 중앙값 |
| `min_time`, `max_time` | 가장 이른·늦은 봉 시각 |
| `adj_rows_equal` | 수정/무수정 봉 수가 같은지 |
| `bad_price`, `bad_ohlc` | 가격 0 이하인 봉, 고가·저가가 맞지 않는 봉 수 |
| `zero_volume` | 거래량 0인 봉 수 |
| `max_adj_return` | 수정주가 1분 수익률 절댓값 최대 |
| `segment_count`, `segments` | 수정/무수정 비율 구간 수와 구간 (`시작~끝:비율`). 2개 이상이면 그 기간에 권리 변동 |
| `vs_krx_bp` | (NXT만) 같은 분 KRX 종가와 차이 중앙값, bp |

마지막 점검은 시작일 맞추기 **전**에 실행한 결과라, `check_KRX.csv`의 `first`에 2024-10-11이 남아 있고 롯데케미칼(011170)의 `adj_rows_equal`이 `False`입니다. 점검 셀을 다시 돌리면 갱신됩니다.

### 이 노트북이 만들지 않은 폴더

- `creon_1min/snapshots/20261102/`: 2026-11-02 장중에 받은 반도체 15종목 KRX 분봉입니다 (2024-10-11~2026-10-30, 날짜 이름으로 저장하던 이전 버전). 겹치는 봉은 `all_stocks`와 값이 모두 같습니다.
- `creon_1min/probe/`: `night_nxt_probe.ipynb` 등 조회 범위를 점검한 노트북의 결과입니다.

In [9]:
import ctypes
import json
import platform
import shutil
import struct
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import win32com.client

## 0. 설정

In [10]:
# ============================================================
# 수집 대상
# ============================================================

MARKETS = {
    # GetStockListByMarket 시장 구분값: 시장명
    1: "KOSPI",
    2: "KOSDAQ",
}

# GetStockSectionKind 부 구분값 중 주권. ETF(10), ETN(17), 리츠(3), 외국주권(13) 등은 제외
SECTION_STOCK = 1

# ============================================================
# 요청 설정
# ============================================================

# StockChart 필드 번호 -> 컬럼명 (응답은 필드 번호 오름차순으로 온다)
FIELDS = {
    0: "date",
    1: "time",
    2: "open",
    3: "high",
    4: "low",
    5: "close",
    8: "volume",
    9: "value",
}

# 실제 한계(약 19만 개)보다 넉넉하게 잡아서, 우리 값이 아니라 서버 쪽 한계에서 멈추게 한다
REQUEST_COUNT = 1_000_000

# 시세 조회 제한 구분값 (0: 주문, 1: 시세, 2: 실시간)
LIMIT_TYPE_QUOTE = 1

# 모든 종목·데이터의 저장 기간. 범위 밖의 봉은 버린다
# 크레온 조회 범위가 날마다 하루씩 밀려서, 수집이 날짜를 넘기며 시작일이 2024-10-11과 10-14로 나뉘었다. 늦은 쪽에 맞춘다
START_DATE = 20241014
END_DATE = 20261102

# KRX 장후 봉(2026-09부터)은 받지 않는다. END_DATE의 마지막 열 시각 전에는 그날 봉이 완성되지 않았으므로 받지 않고 미룬다
DATASETS = {
    # 저장 폴더명: (거래소, KRX 조회구분, 수정주가 여부, END_DATE 봉이 완성되는 시각)
    "adjusted": ("K", "2", True, 1640),  # KRX 정규장 (수능일 폐장 16:30 기준)
    "raw": ("K", "2", False, 1640),
    "nxt_adjusted": ("N", "1", True, 2010),  # NXT 프리·메인·애프터마켓 (08:00~20:00)
    "nxt_raw": ("N", "1", False, 2010),
}

# 결과 점검에서 수정/무수정 짝으로 묶는 단위
DATASET_FAMILIES = {
    "KRX": ("adjusted", "raw"),
    "NXT": ("nxt_adjusted", "nxt_raw"),
}

# 실패가 이만큼 연달아 나면 연결 문제로 보고 멈춘다
MAX_CONSECUTIVE_FAILURES = 10

# 디스크 여유가 이보다 적으면 경고 (15종목 실측 파일당 약 2MB, 전 종목 약 10~15GB 예상)
MIN_FREE_GB = 30

# ============================================================
# 저장 경로
# ============================================================

DATA_DIR = Path("creon_1min")

# 여러 날에 걸쳐 이어 받으므로 날짜가 아닌 고정 이름을 쓴다. 처음부터 새로 받으려면 이름을 바꾼다
SNAPSHOT_DIR = DATA_DIR / "snapshots" / "all_stocks"
UNIVERSE_PATH = SNAPSHOT_DIR / "universe.csv"
MANIFEST_PATH = SNAPSHOT_DIR / "manifest.jsonl"
REPORT_PATH = SNAPSHOT_DIR / "collection_report.txt"

## 0-1. 공통 유틸 (리포트, 매니페스트)

In [11]:
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# 받은 시세 데이터가 git에 올라가지 않도록 폴더 전체를 제외
(DATA_DIR / ".gitignore").write_text("*\n")


def report(*args):
    # 화면 출력과 동시에 파일에도 남긴다. 여러 번 나눠 실행하므로 이어서 기록한다
    line = " ".join(str(arg) for arg in args)

    print(line)

    with open(REPORT_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path):
    path = Path(path)

    if not path.exists():
        return []

    records = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


report(f"\n===== 실행 {datetime.now():%Y-%m-%d %H:%M} =====")


===== 실행 2026-11-03 16:54 =====


## 1. 실행 환경 점검

In [12]:
bits = struct.calcsize("P") * 8
is_admin = bool(ctypes.windll.shell32.IsUserAnAdmin())
free_gb = shutil.disk_usage(SNAPSHOT_DIR.resolve()).free / 1024**3

report("Python:", platform.python_version(), f"{bits}bit")
report("pandas:", pd.__version__)
report("관리자 권한:", is_admin)
report(f"디스크 여유: {free_gb:.1f}GB")

if free_gb < MIN_FREE_GB:
    report(f"경고: 디스크 여유가 {MIN_FREE_GB}GB보다 적습니다. 수집 중 공간이 부족할 수 있습니다.")

# 크레온 COM 객체는 32비트 전용이라 64비트 Python에서는 객체 생성부터 실패한다
if bits != 32:
    raise RuntimeError("32비트 Python으로 실행해야 합니다.")

if not is_admin:
    report("경고: 관리자 권한이 아닙니다. 아래 연결에서 실패하면 VS Code를 관리자 권한으로 다시 실행하세요.")

cybos = win32com.client.Dispatch("CpUtil.CpCybos")

report("크레온 연결:", cybos.IsConnect)
report("시세 조회 잔여 횟수:", cybos.GetLimitRemainCount(LIMIT_TYPE_QUOTE))

if cybos.IsConnect != 1:
    raise RuntimeError("크레온 플러스에 연결되어 있지 않습니다. 관리자 권한으로 실행해 로그인했는지, 윈도우 화면에서 실행 중인지 확인하세요.")

Python: 3.11.9 32bit
pandas: 2.0.3
관리자 권한: True
디스크 여유: 322.6GB
크레온 연결: 1
시세 조회 잔여 횟수: 60


## 2. 종목 목록

처음 실행할 때 크레온에서 KOSPI·KOSDAQ 전 종목을 받아 `universe.csv`로 저장하고, 이후에는 이 파일을 그대로 씁니다. 나눠 실행하는 동안 대상이 바뀌지 않게 하기 위해서입니다.

**보통주 판정** (크레온에 우선주·스팩 구분 함수가 없어 규칙으로 판정)
- 부 구분이 주권(1)
- 종목코드 끝자리가 `0` (우선주는 5, 7, 9, K 등)
- 종목명에 `스팩`이 없음

판정 결과는 `common` 열에 남으므로, 아래 출력에서 걸러진 종목이 맞는지 확인하세요. 현재 상장된 종목만 대상이라 상장폐지 종목은 없습니다.

In [13]:
code_mgr = win32com.client.Dispatch("CpUtil.CpCodeMgr")


def build_universe():
    rows = []

    for market_kind, market in MARKETS.items():
        for full_code in code_mgr.GetStockListByMarket(market_kind):
            rows.append({
                "code": full_code[1:],
                "name": code_mgr.CodeToName(full_code),
                "market": market,
                "section": code_mgr.GetStockSectionKind(full_code),
                "listed_date": code_mgr.GetStockListedDate(full_code),
                "status": code_mgr.GetStockStatusKind(full_code),  # 0 정상, 1 거래정지, 2 거래중단
                "nxt": int(bool(code_mgr.IsNxtTrdPsbl(full_code))),
            })

    universe = pd.DataFrame(rows)

    universe["common"] = (
        (universe["section"] == SECTION_STOCK)
        & universe["code"].str.endswith("0")
        & ~universe["name"].str.contains("스팩")
    )

    return universe


if UNIVERSE_PATH.exists():
    universe = pd.read_csv(UNIVERSE_PATH, dtype={"code": str})
    report(f"종목 목록: 저장된 파일 사용 ({UNIVERSE_PATH})")
else:
    universe = build_universe()
    universe.to_csv(UNIVERSE_PATH, index=False, encoding="utf-8")
    report(f"종목 목록: 크레온에서 새로 받아 저장 ({UNIVERSE_PATH})")

targets = universe[universe["common"]].reset_index(drop=True)
stocks = universe[universe["section"] == SECTION_STOCK]

report(f"전체 {len(universe):,}개 중 보통주 {len(targets):,}개 (NXT 거래 가능 {int(targets['nxt'].sum()):,}개)")
report("시장별 보통주:", targets["market"].value_counts().to_dict())
report("부 구분별 전체 종목 수:", universe["section"].value_counts().sort_index().to_dict())
report("주권 중 제외된 종목 예시 (우선주·스팩):", stocks[~stocks["common"]]["name"].head(15).tolist())
report("보통주로 남은 종목 중 이름이 '우'로 끝나는 종목 (판정 확인용):", targets[targets["name"].str.endswith(("우", "우B", "우C"))]["name"].tolist())

종목 목록: 저장된 파일 사용 (creon_1min\snapshots\all_stocks\universe.csv)
전체 4,309개 중 보통주 2,539개 (NXT 거래 가능 603개)
시장별 보통주: {'KOSDAQ': 1738, 'KOSPI': 801}
부 구분별 전체 종목 수: {1: 2719, 2: 1, 3: 23, 5: 2, 6: 10, 10: 1183, 13: 12, 17: 357, 18: 2}
주권 중 제외된 종목 예시 (우선주·스팩): ['하이트진로2우B', '유한양행우', '하이트진로홀딩스우', '두산우', '두산2우B', 'DL우', '유유제약1우', '유유제약2우B', '노루홀딩스우', '흥국화재우', '현대건설우', '삼성화재우', '한화3우B', 'CJ우', 'CJ4우(전환)']
보통주로 남은 종목 중 이름이 '우'로 끝나는 종목 (판정 확인용): ['에코글로우', '이오플로우', '성우']


## 3. 공통 요청 함수

분봉은 기간 요청을 쓸 수 없어서(공식 문서) **개수 요청 + 연속 조회**로 받습니다. 응답은 최신 봉부터 옵니다.

개수 요청에서는 `Continue` 값이 끝까지 1로 남을 수 있어, 더 과거 데이터가 안 나오는 경우도 종료 조건에 넣었습니다. 어떤 조건으로 끝났는지는 `stop_reason`에 남습니다.

In [14]:
def wait_for_quote_limit():
    # 시세 조회 제한에 걸리기 직전이면 제한이 풀릴 때까지 기다린다
    while cybos.GetLimitRemainCount(LIMIT_TYPE_QUOTE) <= 1:
        time.sleep(cybos.LimitRequestRemainTime / 1000 + 0.05)


def build_chart(code, adjusted, exchange="K", krx_session="1", count=REQUEST_COUNT):
    chart = win32com.client.Dispatch("CpSysDib.StockChart")

    chart.SetInputValue(0, f"A{code}")
    chart.SetInputValue(1, ord("2"))  # 개수 요청
    chart.SetInputValue(4, count)
    chart.SetInputValue(5, list(FIELDS))
    chart.SetInputValue(6, ord("m"))  # 분봉
    chart.SetInputValue(7, 1)  # 1분 주기
    chart.SetInputValue(9, ord("1" if adjusted else "0"))
    chart.SetInputValue(12, ord(exchange))  # A: 전체, K: KRX, N: NXT
    chart.SetInputValue(13, ord(krx_session))  # 1: 애프터마켓 포함, 2: 정규장만

    return chart


def request_page(chart):
    wait_for_quote_limit()

    chart.BlockRequest()

    status = chart.GetDibStatus()

    if status != 0:
        raise RuntimeError(f"요청 실패 (status={status}): {chart.GetDibMsg1()}")

    received = chart.GetHeaderValue(3)

    # GetDataValue의 첫 인자는 필드 번호가 아니라 요청한 필드 안에서의 순서
    rows = [
        [chart.GetDataValue(column, row) for column in range(len(FIELDS))]
        for row in range(received)
    ]

    return pd.DataFrame(rows, columns=list(FIELDS.values()))


def bar_key(df):
    return df["date"].astype("int64") * 10000 + df["time"].astype("int64")


def fetch_all_bars(code, adjusted, exchange="K", krx_session="1", count=REQUEST_COUNT):
    chart = build_chart(code, adjusted, exchange, krx_session, count)

    pages = []
    received_total = 0
    oldest_key = None

    while True:
        page = request_page(chart)

        if page.empty:
            stop_reason = "empty_page"
            break

        page_oldest_key = bar_key(page).min()

        if oldest_key is not None and page_oldest_key >= oldest_key:
            stop_reason = "no_older_data"
            break

        oldest_key = page_oldest_key
        pages.append(page)
        received_total += len(page)

        if len(pages) % 20 == 0:
            print(f"  요청 {len(pages)}회, 누적 {received_total:,}개, 가장 오래된 봉 {oldest_key}")

        if not chart.Continue:
            stop_reason = "continue_0"
            break

        if received_total >= count:
            stop_reason = "request_count_reached"
            break

    bars = pd.concat(pages, ignore_index=True) if pages else pd.DataFrame(columns=list(FIELDS.values()))
    bars = bars.drop_duplicates(["date", "time"]).sort_values(["date", "time"], ignore_index=True)

    info = {
        "requests": len(pages),
        "received": received_total,
        "stop_reason": stop_reason,
        "max_bars_per_request": max((len(page) for page in pages), default=0),
    }

    return bars, info


def summarize_bars(bars):
    if bars.empty:
        return {"rows": 0}

    bars_per_day = bars.groupby("date").size()
    value_unit = (bars["value"] / (bars["close"] * bars["volume"]).where(lambda s: s > 0)).median()

    return {
        "rows": len(bars),
        "first": int(bar_key(bars).min()),
        "last": int(bar_key(bars).max()),
        "trading_days": int(bars["date"].nunique()),
        "median_bars_per_day": float(bars_per_day.median()),
        "min_time": int(bars["time"].min()),
        "max_time": int(bars["time"].max()),
        "value_over_close_x_volume": round(float(value_unit), 6),
    }

## 4. 전 종목 수집

15종목 점검에서 확인한 내용을 반영했습니다.
- KRX 조회 한계: 2024-10-11부터 약 2년. 서버가 `Continue=0`으로 끝을 알려줌
- NXT: 삼성전자 기준 2025-03-24부터 08:01~20:00, 하루 690개. 같은 분 KRX 종가와 중앙값 차이 0bp
- 수정주가는 분봉에도 적용됨. 다만 수정주가는 가격과 거래량이 반올림되고 이후 권리 변동에 따라 값이 바뀌므로 **무수정주가도 함께 저장**
- 시세 조회 제한은 15초당 60회, 요청당 최대 2,499개. 종목·데이터당 약 20초

**진행 방식**
- 전 종목 KRX(종목마다 수정·무수정)를 먼저 받고, 그다음 NXT 거래 가능 종목의 NXT(수정·무수정)를 받습니다.
- `END_DATE` 봉이 아직 완성되지 않은 시각이면(KRX 16:40, NXT 20:10 전) 그 작업은 받지 않고 미룹니다. 끝에 미룬 개수가 나오니, 그 시각 이후에 다시 실행하면 됩니다.
- 파일은 다 쓴 뒤에 이름을 바꾸므로, 중간에 끊겨도 불완전한 파일이 완료 파일로 남지 않습니다.
- 거래가 없어 봉이 하나도 없는 경우도 빈 파일로 저장해 완료로 처리합니다.
- 20개마다 진행률과 남은 시간 추정치를 보여줍니다.

In [15]:
def snapshot_path(kind, code):
    return SNAPSHOT_DIR / kind / f"{code}.csv.gz"


def end_date_finished(finished_hhmm):
    """END_DATE 봉이 완성됐는지 (그날 마지막 봉 시각이 지났는지)"""
    now = datetime.now()
    today = int(now.strftime("%Y%m%d"))

    return today > END_DATE or (today == END_DATE and now.hour * 100 + now.minute >= finished_hhmm)


def save_bars(bars, path):
    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = path.with_name(path.name + ".tmp")

    bars.to_csv(temp_path, index=False, compression="gzip")
    temp_path.replace(path)


# KRX를 전 종목 먼저 받고 NXT를 나중에 받는다. 종목 안에서는 수정·무수정을 이어서 받는다
jobs = [
    (row["code"], row["name"], kind)
    for exchange_order in ["K", "N"]
    for row in targets.to_dict("records")
    for kind, (exchange, krx_session, adjusted, finished_hhmm) in DATASETS.items()
    if exchange == exchange_order and (exchange != "N" or row["nxt"])
]
pending = [job for job in jobs if not snapshot_path(job[2], job[0]).exists()]

report(f"\n전체 작업 {len(jobs):,}개, 완료 {len(jobs) - len(pending):,}개, 남은 작업 {len(pending):,}개")

failures = []
deferred = 0
consecutive_failures = 0
started_at = time.time()

for number, (code, name, kind) in enumerate(pending, start=1):
    if cybos.IsConnect != 1:
        report("크레온 연결이 끊겼습니다. 다시 로그인한 뒤 이 셀을 다시 실행하세요.")
        break

    exchange, krx_session, adjusted, finished_hhmm = DATASETS[kind]

    if not end_date_finished(finished_hhmm):
        deferred += 1
        continue

    job_started_at = time.time()

    try:
        bars, info = fetch_all_bars(code, adjusted, exchange, krx_session)
    except RuntimeError as error:
        failures.append((code, name, kind, str(error)))
        consecutive_failures += 1
        report(f"실패 {code} {name} {kind}: {error}")

        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            report(f"실패가 {consecutive_failures}번 연달아 나서 멈춥니다. 크레온 연결을 확인한 뒤 이 셀을 다시 실행하세요.")
            break

        continue

    consecutive_failures = 0
    bars = bars[(bars["date"] >= START_DATE) & (bars["date"] <= END_DATE)]

    save_bars(bars, snapshot_path(kind, code))

    info.update(
        code=code,
        kind=kind,
        rows=len(bars),
        first_date=int(bars["date"].min()) if len(bars) else None,
        last_date=int(bars["date"].max()) if len(bars) else None,
        seconds=round(time.time() - job_started_at, 1),
        saved_at=datetime.now().isoformat(timespec="seconds"),
    )
    append_jsonl(MANIFEST_PATH, info)

    report(f"완료 {code} {name} {kind}: {len(bars):,}개, {info['first_date']}~{info['last_date']}, {info['stop_reason']}")

    if (number - deferred) % 20 == 0:
        seconds_per_job = (time.time() - started_at) / (number - deferred)
        remaining_hours = seconds_per_job * (len(pending) - number) / 3600
        report(f"  진행 {number:,}/{len(pending):,}, 작업당 {seconds_per_job:.1f}초, 남은 시간 약 {remaining_hours:.1f}시간")

report(f"\n이번 실행 실패 {len(failures)}건:", failures[:30])

if deferred:
    report(f"END_DATE({END_DATE}) 봉이 아직 완성되지 않아 미룬 작업 {deferred:,}개. 장 마감 뒤(KRX 16:40, NXT 20:10 이후)에 다시 실행하세요.")


전체 작업 6,284개, 완료 6,283개, 남은 작업 1개
  요청 20회, 누적 49,980개, 가장 오래된 봉 202607160916
  요청 40회, 누적 99,960개, 가장 오래된 봉 202603311304
  요청 60회, 누적 149,940개, 가장 오래된 봉 202512081735
  요청 80회, 누적 199,920개, 가장 오래된 봉 202508211007
  요청 100회, 누적 249,900개, 가장 오래된 봉 202505021655
완료 035420 NAVER nxt_raw: 267,960개, 20250324~20261102, continue_0

이번 실행 실패 0건: []


## 5. 수집 결과 점검

저장된 파일을 다시 읽어 KRX와 NXT를 각각 점검합니다. 종목이 많으므로 전체 점검표는 `check_KRX.csv`, `check_NXT.csv`로 저장하고, 리포트에는 요약과 이상 종목만 남깁니다. 파일 수가 많아 수십 분 걸릴 수 있습니다.

- **수정/무수정 비율 구간**: 하루 중앙값으로 구간을 나눠 분할 같은 권리 변동을 보여줍니다 (`segments`).
- **NXT**: 같은 분의 KRX 종가와 차이(`vs_krx_bp`, 중앙값)도 봅니다.

In [16]:
def adjustment_segments(adjusted_bars, raw_bars):
    merged = adjusted_bars.merge(raw_bars, on=["date", "time"], suffixes=("_adj", "_raw"))
    merged["ratio"] = merged["close_adj"] / merged["close_raw"]

    daily_ratio = merged.groupby("date")["ratio"].median().round(3)
    segment_id = (daily_ratio != daily_ratio.shift()).cumsum()

    segments = daily_ratio.groupby(segment_id).agg(lambda s: f"{s.index[0]}~{s.index[-1]}:{s.iloc[0]}")

    return list(segments)


manifest = {(record["code"], record["kind"]): record for record in load_jsonl(MANIFEST_PATH)}

for family, (adjusted_kind, raw_kind) in DATASET_FAMILIES.items():
    checks = []

    for row in targets.to_dict("records"):
        code = row["code"]

        if not (snapshot_path(adjusted_kind, code).exists() and snapshot_path(raw_kind, code).exists()):
            continue

        adjusted_bars = pd.read_csv(snapshot_path(adjusted_kind, code))
        raw_bars = pd.read_csv(snapshot_path(raw_kind, code))

        check = {
            "code": code,
            "name": row["name"],
            "market": row["market"],
            "stop": manifest.get((code, raw_kind), {}).get("stop_reason"),
            "requests": manifest.get((code, raw_kind), {}).get("requests"),
            "rows": len(raw_bars),
        }

        if raw_bars.empty:
            checks.append(check)
            continue

        prices = raw_bars[["open", "high", "low", "close"]]
        summary = summarize_bars(raw_bars)
        segments = adjustment_segments(adjusted_bars, raw_bars)

        check.update({
            "first": summary["first"],
            "last": summary["last"],
            "days": summary["trading_days"],
            "median_bars": summary["median_bars_per_day"],
            "min_time": summary["min_time"],
            "max_time": summary["max_time"],
            "adj_rows_equal": len(adjusted_bars) == len(raw_bars),
            "bad_price": int(((prices <= 0).any(axis=1)).sum()),
            "bad_ohlc": int(((prices["high"] < prices.max(axis=1)) | (prices["low"] > prices.min(axis=1))).sum()),
            "zero_volume": int((raw_bars["volume"] == 0).sum()),
            "max_adj_return": round(float(adjusted_bars["close"].pct_change().abs().max()), 3),
            "segment_count": len(segments),
            "segments": " ".join(segments),
        })

        if family == "NXT" and snapshot_path("raw", code).exists():
            krx_bars = pd.read_csv(snapshot_path("raw", code))
            overlap = raw_bars.merge(krx_bars, on=["date", "time"], suffixes=("_nxt", "_krx"))

            if len(overlap):
                check["vs_krx_bp"] = round(float((overlap["close_nxt"] / overlap["close_krx"] - 1).abs().median() * 10000), 2)

        checks.append(check)

    if not checks:
        report(f"\n[{family} 점검] 받은 파일 없음")
        continue

    table = pd.DataFrame(checks)
    table.to_csv(SNAPSHOT_DIR / f"check_{family}.csv", index=False, encoding="utf-8")

    report(f"\n[{family} 점검] 파일 있는 종목 {len(table):,}개, 봉 합계 {int(table['rows'].sum()):,}개")

    has_bars = table[table["rows"] > 0]

    report("봉 없는 종목:", int((table["rows"] == 0).sum()))
    report("종료 사유:", table["stop"].value_counts().to_dict())
    report("시작 날짜 분포 (상위 5):", (has_bars["first"] // 10000).astype(int).value_counts().head(5).to_dict())
    report("권리 변동(비율 구간 2개 이상) 종목:", int((has_bars["segment_count"] > 1).sum()))

    problems = has_bars[
        (has_bars["bad_price"] > 0)
        | (has_bars["bad_ohlc"] > 0)
        | ~has_bars["adj_rows_equal"].astype(bool)
        | (has_bars["stop"] != "continue_0")
    ]

    report(f"이상 종목 {len(problems)}개 (가격 오류, 고저 오류, 수정/무수정 봉 수 불일치, 비정상 종료):")

    if len(problems):
        report(problems[["code", "name", "rows", "stop", "bad_price", "bad_ohlc", "adj_rows_equal"]].head(30).to_string(index=False))

    if "vs_krx_bp" in has_bars:
        report("NXT-KRX 종가 차이(bp) 분포:", has_bars["vs_krx_bp"].describe().round(2).to_dict())

report(f"\n리포트 저장 위치: {REPORT_PATH.resolve()}")


[KRX 점검] 파일 있는 종목 2,539개, 봉 합계 295,887,762개
봉 없는 종목: 15
종료 사유: {'continue_0': 2523, 'empty_page': 16}
시작 날짜 분포 (상위 5): {20241014: 1997, 20241011: 333, 20241101: 5, 20250509: 3, 20250320: 2}
권리 변동(비율 구간 2개 이상) 종목: 519
이상 종목 2개 (가격 오류, 고저 오류, 수정/무수정 봉 수 불일치, 비정상 종료):
  code  name   rows       stop  bad_price  bad_ohlc adj_rows_equal
011170 롯데케미칼 189738 continue_0        0.0       0.0          False
039490  키움증권 189924 empty_page        0.0       0.0           True

[NXT 점검] 파일 있는 종목 603개, 봉 합계 87,422,243개
봉 없는 종목: 0
종료 사유: {'continue_0': 603}
시작 날짜 분포 (상위 5): {20250324: 187, 20250331: 182, 20250317: 96, 20260102: 87, 20250701: 31}
권리 변동(비율 구간 2개 이상) 종목: 52
이상 종목 0개 (가격 오류, 고저 오류, 수정/무수정 봉 수 불일치, 비정상 종료):
NXT-KRX 종가 차이(bp) 분포: {'count': 603.0, 'mean': 6.27, 'std': 4.56, 'min': 0.0, '25%': 0.0, '50%': 6.4, '75%': 9.95, 'max': 20.75}

리포트 저장 위치: C:\Users\SD1-12\Desktop\oneminbar\creon_1min\snapshots\all_stocks\collection_report.txt


## 6. 시작일 맞추기 (단독 실행)

크레온 조회 범위가 날마다 하루씩 밀려서, 수집이 날짜를 넘기며 KRX 시작일이 2024-10-11과 10-14로 나뉘었습니다. 저장된 파일에서 `START_DATE` 이전 봉을 지워 시작일을 맞춥니다.

- **이 셀만 실행하면 됩니다.** 크레온 연결이나 앞 셀 실행이 필요 없습니다.
- 파일은 날짜순으로 저장되어 있으므로 첫 줄만 읽어 지울 봉이 있는지 판단하고, 해당 파일만 다시 씁니다. 다시 실행해도 바뀌는 것이 없습니다.
- 끝에 파일별 첫 날짜 분포를 보여주므로, 점검 셀을 다시 돌리지 않아도 결과를 확인할 수 있습니다.
- 지운 봉은 되돌릴 수 없습니다.

In [17]:
import gzip
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd

START_DATE = 20241014
SNAPSHOT_DIR = Path("creon_1min") / "snapshots" / "all_stocks"
KINDS = ["adjusted", "raw", "nxt_adjusted", "nxt_raw"]
REPORT_PATH = SNAPSHOT_DIR / "collection_report.txt"


def log(*args):
    # 화면 출력과 동시에 수집 리포트에도 이어서 기록한다
    line = " ".join(str(arg) for arg in args)

    print(line)

    with open(REPORT_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def first_bar_date(path):
    """날짜순으로 저장된 파일의 첫 봉 날짜. 봉이 없으면 None"""
    with gzip.open(path, "rt", encoding="utf-8") as f:
        f.readline()
        line = f.readline()

    return int(line.split(",")[0]) if line.strip() else None


def save_bars(bars, path):
    temp_path = path.with_name(path.name + ".tmp")

    bars.to_csv(temp_path, index=False, compression="gzip")
    temp_path.replace(path)


log(f"\n===== 시작일 맞추기 {datetime.now():%Y-%m-%d %H:%M}: {START_DATE} 이전 봉 삭제 =====")

for kind in KINDS:
    paths = sorted((SNAPSHOT_DIR / kind).glob("*.csv.gz"))
    trimmed_files = 0
    removed_rows = 0

    for path in paths:
        first_date = first_bar_date(path)

        if first_date is None or first_date >= START_DATE:
            continue

        bars = pd.read_csv(path)
        kept = bars[bars["date"] >= START_DATE]

        save_bars(kept, path)

        trimmed_files += 1
        removed_rows += len(bars) - len(kept)

    first_dates = Counter(first_bar_date(path) for path in paths)

    log(f"{kind}: 파일 {len(paths):,}개 중 {trimmed_files:,}개에서 봉 {removed_rows:,}개 삭제")
    log(f"  첫 날짜 분포 (상위 5): {first_dates.most_common(5)}")



===== 시작일 맞추기 2026-11-03 17:41: 20241014 이전 봉 삭제 =====
adjusted: 파일 2,539개 중 334개에서 봉 81,443개 삭제
  첫 날짜 분포 (상위 5): [(20241014, 2330), (None, 15), (20241101, 5), (20250509, 3), (20260114, 2)]
raw: 파일 2,539개 중 333개에서 봉 81,062개 삭제
  첫 날짜 분포 (상위 5): [(20241014, 2330), (None, 15), (20241101, 5), (20250509, 3), (20260114, 2)]
nxt_adjusted: 파일 603개 중 0개에서 봉 0개 삭제
  첫 날짜 분포 (상위 5): [(20250324, 187), (20250331, 182), (20250317, 96), (20260102, 87), (20250701, 31)]
nxt_raw: 파일 603개 중 0개에서 봉 0개 삭제
  첫 날짜 분포 (상위 5): [(20250324, 187), (20250331, 182), (20250317, 96), (20260102, 87), (20250701, 31)]
